<a href="https://colab.research.google.com/github/1Bur1/clothes-image-Classifier/blob/main/01_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2 — Build & Train the Fashion-MNIST CNN

**Project:** Fashion Item Image Classifier  
**Dataset:** Fashion-MNIST (Zalando Research, MIT License)  
**Goal:** Train a CNN to classify 10 clothing categories from 28×28 grayscale images.

---

## Notebook Structure
1. Setup & Imports
2. `FashionClassifier` Class Definition
3. Exploratory Data Analysis (EDA)
4. Experiment 1 — Baseline CNN
5. Experiment 2 — Add Dropout
6. Experiment 3 — More Epochs (Final Model)
7. Experiment Summary Table
8. Sample Predictions
9. Save Best Model

## 1. Setup & Imports

We fix **random seeds** before anything else so every run produces identical weights and results.  
This is required for reproducibility — two engineers running this notebook must get the same numbers.

| Library | Purpose |
|---|---|
| `numpy` | Array operations and random seed |
| `tensorflow / keras` | Building and training the CNN |
| `matplotlib` | Plotting training curves and sample images |
| `os` | Creating output folders |

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Fix seeds for full reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Create output folders if they do not exist yet
os.makedirs('../models',  exist_ok=True)
os.makedirs('../reports', exist_ok=True)

print(f'TensorFlow version : {tf.__version__}')
print('Seeds set. Output folders ready.')

## 2. `FashionClassifier` Class Definition

All project logic lives in a single **class** with clearly separated methods.  
This is how production ML code is structured — each method does one job, making the code easy to read, test, and extend.

### Class Methods Overview

| Method | What it does |
|---|---|
| `__init__` | Stores class names and resets seeds |
| `load_data(val_size, train_size)` | Loads Fashion-MNIST, normalises, adds channel dim, splits |
| `show_class_distribution()` | Bar chart of samples per class (EDA) |
| `show_samples()` | Plots one image per class (EDA) |
| `build_model(dropout)` | Builds the CNN architecture |
| `train(epochs, batch_size)` | Compiles and trains the model |
| `plot_curves(title, save_path)` | Plots accuracy and loss training curves |
| `predict_samples(images, labels, n)` | Prints human-readable predictions using class names |
| `save(path)` | Saves the trained model to disk |

In [ ]:
class FashionClassifier:
    """
    End-to-end CNN classifier for the Fashion-MNIST dataset.
    Encapsulates data loading, model building, training, visualisation, and saving.
    """

    # Human-readable names for the 10 Fashion-MNIST categories
    CLASS_NAMES = [
        'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
        'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
    ]

    def __init__(self, seed=42):
        """
        Initialise the classifier.
        Resets random seeds so every instance produces reproducible results.
        """
        self.seed = seed
        np.random.seed(seed)
        tf.random.set_seed(seed)

        # These attributes are populated by load_data() and build_model()
        self.X_train = self.X_val = self.X_test = None
        self.y_train = self.y_val = self.y_test = None
        self.model   = None
        self.history = None

        print(f'FashionClassifier initialised (seed={seed})')

    # ------------------------------------------------------------------
    # DATA
    # ------------------------------------------------------------------

    def load_data(self):
        """
        Load all 70,000 Fashion-MNIST images and split 70 / 15 / 15.

        Steps:
          1. Combine built-in train (60k) + test (10k) = 70,000 images
          2. Normalise pixels [0,255] -> [0.0,1.0]
          3. Add channel dim (28,28) -> (28,28,1)
          4. Shuffle with fixed seed for reproducibility
          5. Split: 70% train / 15% val / 15% test
        """
        (X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()

        # Step 1 -- combine all 70,000 images
        X_all = np.concatenate([X_tr, X_te], axis=0)
        y_all = np.concatenate([y_tr, y_te], axis=0)

        # Step 2 & 3 -- normalise + channel dim
        X_all = (X_all / 255.0).reshape(-1, 28, 28, 1)

        # Step 4 -- shuffle with fixed seed
        np.random.seed(self.seed)
        idx   = np.random.permutation(len(X_all))
        X_all = X_all[idx]
        y_all = y_all[idx]

        # Step 5 -- split 70 / 15 / 15
        n      = len(X_all)            # 70,000
        n_test = int(n * 0.15)         # 10,500
        n_val  = int(n * 0.15)         # 10,500
        # train = 70,000 - 10,500 - 10,500 = 49,000

        self.X_test,  self.y_test  = X_all[:n_test],              y_all[:n_test]
        self.X_val,   self.y_val   = X_all[n_test:n_test+n_val],  y_all[n_test:n_test+n_val]
        self.X_train, self.y_train = X_all[n_test+n_val:],        y_all[n_test+n_val:]

        print('Data loaded (full 70,000 images, 70/15/15 split):')
        print(f'  Training   : {self.X_train.shape}  ({len(self.X_train):,} images - 70%)')
        print(f'  Validation : {self.X_val.shape}  ({len(self.X_val):,} images - 15%)')
        print(f'  Test       : {self.X_test.shape}  ({len(self.X_test):,} images - 15%) <- locked until Phase 3')
        return self
    def show_class_distribution(self, save_path='../reports/class_distribution.png'):
        """
        Bar chart of training samples per class.
        Balanced classes mean accuracy is a trustworthy metric.
        """
        _, counts = np.unique(self.y_train, return_counts=True)

        fig, ax = plt.subplots(figsize=(10, 4))
        bars = ax.bar(self.CLASS_NAMES, counts, color='steelblue', edgecolor='white')
        ax.set_title('Training Set — Samples per Class', fontsize=14)
        ax.set_xlabel('Class')
        ax.set_ylabel('Number of Images')
        ax.set_ylim(0, max(counts) * 1.2)
        plt.xticks(rotation=30, ha='right')

        for bar, count in zip(bars, counts):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
                    str(count), ha='center', va='bottom', fontsize=9)

        plt.tight_layout()
        plt.savefig(save_path, dpi=150)
        plt.show()
        print(f'Saved -> {save_path}')
        return self

    def show_samples(self, save_path='../reports/sample_images.png'):
        """
        Display one example image per class.
        Reveals visually similar classes (Shirt vs T-shirt, Coat vs Pullover)
        that are the main source of model errors.
        """
        fig, axes = plt.subplots(2, 5, figsize=(12, 5))
        fig.suptitle('One Sample Image per Class', fontsize=14)

        for class_idx, ax in enumerate(axes.flat):
            sample_idx = np.where(self.y_train == class_idx)[0][0]
            ax.imshow(self.X_train[sample_idx].squeeze(), cmap='gray')
            ax.set_title(self.CLASS_NAMES[class_idx], fontsize=9)
            ax.axis('off')

        plt.tight_layout()
        plt.savefig(save_path, dpi=150)
        plt.show()
        print(f'Saved -> {save_path}')
        return self

    # ------------------------------------------------------------------
    # MODEL
    # ------------------------------------------------------------------

    def build_model(self, dropout=None):
        """
        Build CNN with Keras Sequential API.

        Architecture:
          Conv2D(32) -> MaxPool -> Conv2D(64) -> MaxPool -> Flatten
          -> Dense(128) -> [optional Dropout] -> Dense(10, softmax)

        Parameters
        ----------
        dropout : float or None
            Dropout rate inserted before the output layer to reduce overfitting.
        """
        np.random.seed(self.seed)
        tf.random.set_seed(self.seed)

        model_layers = [
            layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
            layers.MaxPooling2D((2, 2)),
            layers.Conv2D(64, (3, 3), activation='relu'),
            layers.MaxPooling2D((2, 2)),
            layers.Flatten(),
            layers.Dense(128, activation='relu'),
        ]

        if dropout is not None:
            model_layers.append(layers.Dropout(dropout))

        model_layers.append(layers.Dense(10, activation='softmax'))

        self.model = keras.Sequential(model_layers, name='fashion_cnn')
        print(f'Model built (dropout={dropout})')
        self.model.summary()
        return self

    # ------------------------------------------------------------------
    # TRAINING
    # ------------------------------------------------------------------

    def train(self, epochs=5, batch_size=32):
        """
        Compile and train the model.
        Validation accuracy is monitored each epoch to detect overfitting.

        Optimizer : Adam  — adaptive learning rate, works well out of the box
        Loss      : Sparse Categorical Crossentropy — for integer labels 0-9
        """
        self.model.compile(
            optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )

        print(f'Training for {epochs} epochs...')
        self.history = self.model.fit(
            self.X_train, self.y_train,
            validation_data=(self.X_val, self.y_val),
            epochs=epochs,
            batch_size=batch_size,
            verbose=1
        )

        best_val = max(self.history.history['val_accuracy'])
        print(f'Best validation accuracy: {best_val:.4f}  ({best_val*100:.2f}%)')
        return self

    # ------------------------------------------------------------------
    # VISUALISATION
    # ------------------------------------------------------------------

    def plot_curves(self, title='Training Curves', save_path=None):
        """
        Plot accuracy and loss for training and validation sets.
        A large gap between the two lines = overfitting.
        """
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        fig.suptitle(title, fontsize=14)

        for ax, metric in zip(axes, ['accuracy', 'loss']):
            ax.plot(self.history.history[metric],          label='Train',      color='steelblue')
            ax.plot(self.history.history[f'val_{metric}'], label='Validation', color='coral', linestyle='--')
            ax.set_title(metric.capitalize())
            ax.set_xlabel('Epoch')
            ax.set_ylabel(metric.capitalize())
            ax.legend()
            ax.grid(alpha=0.3)

        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=150)
            print(f'Saved -> {save_path}')
        plt.show()
        return self

    # ------------------------------------------------------------------
    # PREDICTION
    # ------------------------------------------------------------------

    def predict_samples(self, images, labels, n=5):
        """
        Print human-readable predictions for the first n images.
        Uses CLASS_NAMES so output shows 'Sneaker' instead of raw number 7.
        """
        preds = self.model.predict(images[:n], verbose=0)
        print(f'{"True Label":<20} {"Predicted":<20} {"Confidence":>10}')
        print('-' * 52)
        for i in range(n):
            true_name  = self.CLASS_NAMES[labels[i]]
            pred_idx   = np.argmax(preds[i])
            pred_name  = self.CLASS_NAMES[pred_idx]
            confidence = preds[i][pred_idx]
            status     = '' if pred_idx == labels[i] else '  WRONG'
            print(f'{true_name:<20} {pred_name:<20} {confidence:>9.1%}{status}')
        return self

    # ------------------------------------------------------------------
    # SAVE
    # ------------------------------------------------------------------

    def save(self, path='../models/fashion_cnn.h5'):
        """
        Save trained model in Keras .h5 format.
        Stores architecture + weights + optimizer state.
        Loaded in 02_evaluate.ipynb for final testing on the test set.
        """
        self.model.save(path)
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f'Model saved -> {path}  ({size_mb:.2f} MB)')
        return self


print('FashionClassifier class defined successfully.')


## 3. Exploratory Data Analysis (EDA)

Before training we explore the data to answer:
1. Are the 10 classes balanced? → determines if accuracy is a trustworthy metric
2. What do the images look like? → reveals visually similar classes that will confuse the model

We create one `FashionClassifier` instance and use it throughout the entire notebook.

In [ ]:
# Create the classifier and load data
clf = FashionClassifier(seed=42)

# Split: 49,000 train (70%) / 10,500 val (15%) / 10,500 test (15%)
clf.load_data()


### 3a. Class Distribution

Each of the 10 classes should have roughly the same number of images.  
A balanced dataset means the model cannot cheat by always guessing the most common class.

In [ ]:
clf.show_class_distribution()

### 3b. Sample Images per Class

Looking at actual images reveals which classes are hardest to distinguish.  
Notice how **Shirt** and **T-shirt/top** look very similar — these will be the main source of errors.

In [ ]:
clf.show_samples()

## 4. Experiment 1 — Baseline CNN

We train the CNN **without Dropout** to establish a reference accuracy.  
Every subsequent experiment must beat this number to justify its change.

**Settings:** no dropout · 5 epochs · batch size 32 · 20,000 training samples (56% of total)

In [ ]:
clf.build_model(dropout=None)
clf.train(epochs=5, batch_size=32)
clf.plot_curves(title='Experiment 1 — Baseline CNN',
                save_path='../reports/training_curves_v1.png')

val_acc_v1 = max(clf.history.history['val_accuracy'])
print(f'Experiment 1 result: {val_acc_v1*100:.2f}%')

## 5. Experiment 2 — Add Dropout

If Experiment 1 shows a gap between train and validation accuracy, the model is **overfitting**.  
Dropout randomly disables 30% of neurons each step, forcing the network to learn more robust patterns.

**Only one change from Experiment 1:** `dropout=0.3`  
Everything else (epochs, batch size, architecture, dataset size) stays identical.

In [ ]:
clf.build_model(dropout=0.3)
clf.train(epochs=5, batch_size=32)
clf.plot_curves(title='Experiment 2 — CNN + Dropout(0.3)',
                save_path='../reports/training_curves_v2.png')

val_acc_v2 = max(clf.history.history['val_accuracy'])
print(f'Experiment 2 result: {val_acc_v2*100:.2f}%')

## 6. Experiment 3 — More Epochs (Final Model)

If validation accuracy is still climbing at epoch 5, training a few more epochs will improve it further.

**Only one change from Experiment 2:** `epochs=8`  
This is the model we save and use for final evaluation in Phase 3.

In [ ]:
clf.build_model(dropout=0.3)
clf.train(epochs=8, batch_size=32)
clf.plot_curves(title='Experiment 3 — CNN + Dropout + 8 Epochs (Final)',
                save_path='../reports/training_curves.png')

val_acc_v3 = max(clf.history.history['val_accuracy'])
print(f'Experiment 3 result: {val_acc_v3*100:.2f}%')

## 7. Experiment Summary Table

We changed **one thing at a time** across three experiments.  
This is the correct scientific method — it tells us exactly which change caused each accuracy improvement.

In [ ]:
print('=' * 62)
print(f'  {"Experiment":<38} {"Best Val Acc":>12}')
print('-' * 62)
print(f'  {"1 — Baseline CNN  (5 epochs, no dropout)":<38} {val_acc_v1*100:>11.2f}%')
print(f'  {"2 — + Dropout(0.3)  (5 epochs)":<38} {val_acc_v2*100:>11.2f}%')
print(f'  {"3 — + Dropout(0.3)  (8 epochs)  <- FINAL":<38} {val_acc_v3*100:>11.2f}%')
print('=' * 62)

## 8. Sample Predictions

We verify the model produces sensible predictions before saving.  
`predict_samples()` uses `CLASS_NAMES` so output shows **'Sneaker'** instead of the raw number **7** —  
immediately understandable without needing to look up the label mapping.

In [ ]:
clf.predict_samples(clf.X_val, clf.y_val, n=8)

## 9. Save Best Model

We save Experiment 3 to `models/fashion_cnn.h5`.  
This file will be loaded in `02_evaluate.ipynb` for final testing on the **untouched test set**.  
Saving means we never need to retrain — the evaluation notebook always uses the exact same weights.

In [ ]:
clf.save('../models/fashion_cnn.h5')
print('\nPhase 2 complete. Proceed to 02_evaluate.ipynb for Phase 3.')